In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.keys import Keys
from selenium.common.exceptions import TimeoutException, NoSuchElementException, StaleElementReferenceException, WebDriverException
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.action_chains import ActionChains
import time
import pandas as pd

In [2]:
# Create options object
options = Options()

options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')
options.add_argument('--disable-infobars')
options.add_argument('--disable-extensions')
options.add_argument('--disable-notifications')
options.add_argument('--window-size=1724,968')
user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/96.0.4664.110 Safari/537.36"
options.add_argument(f'user-agent={user_agent}')

In [3]:
vivino_page = "https://www.vivino.com/explore?e=eJwlzDsKgDAQBNDbTB0_7RaCV7ASkTVGCZhEkvi7vYs284oZxkUq4KwnBcc3FUop6IeaDlqixS71utDJ0ZrMG0KcaTZJI0wPRc7Wr2nU4fAZV-4HWX9UP-VPLZdJ5ANfvskDJxc%3D"

In [4]:
def scroll_down(element):
    #delta_y = element.rect['y']
    ActionChains(driver)\
        .scroll_by_amount(0, 100)\
        .perform()

def scroll_to_element(element):
    ActionChains(driver)\
        .scroll_to_element(element)\
        .perform()

def expand_slider(driver):
    # Locate the range slider container
    slider_container = driver.find_element(By.CLASS_NAME, "rangeSlider__slider--DaZc5")

    # Locate the two slider handles within the container
    handles = slider_container.find_elements(By.CLASS_NAME, 'rc-slider-handle')

    if len(handles) != 2:
        raise ValueError("Expected exactly 2 handles, found: {}".format(len(handles)))

    handle1 = handles[0]
    handle2 = handles[1]

    # Initialize ActionChains
    actions = ActionChains(driver)

    # Get the width of the slider to calculate the movement
    slider = slider_container.find_element(By.CLASS_NAME, 'rc-slider')
    slider_width = slider.size['width']

    # Calculate the offset to move handle1 to the far left
    handle1_x_offset = handle1.location['x'] - slider.location['x']
    # Calculate the offset to move handle2 to the far right
    handle2_x_offset = slider_width - (handle2.location['x'] - slider.location['x'])

    # Drag handle1 to the far left (minimum value)
    actions.click_and_hold(handle1).move_by_offset(-handle1_x_offset, 0).release().perform()

    # Drag handle2 to the far right (maximum value)
    actions.click_and_hold(handle2).move_by_offset(handle2_x_offset, 0).release().perform()

def select_ship_to_country(driver, country):
    try:
        # Locate the "Ship to" dropdown container by checking for the "Ship to" text
        dropdown_container = driver.find_element(By.XPATH, "//div[contains(@class, 'navigationItem_navigationItem__Owbn-')]//span[text()='Ship to']/ancestor::div[contains(@class, 'navigationItem_labelContainer__aZ5-F')]")
        
        # Click the dropdown container to reveal options
        dropdown_container.click()

        # Wait for the dropdown options to be visible
        WebDriverWait(driver, 10).until(
            EC.visibility_of_element_located((By.XPATH, "//div[contains(@class, 'navigationItem_closeWrapper__n28xG')]"))
        )

        # Locate the option for the desired country and click it
        country_option = driver.find_element(By.XPATH, f"//a[@data-shiptotype='Country' and text()='{country}']")
        country_option.click()

        print(f"Successfully selected {country} from the dropdown.")
        time.sleep(4)

    except Exception as e:
        print(f"An error occurred: {e}")

i_start = 0
i_end = 1000
n_to_scroll = i_start + i_end
#number_to_retrieve = 1000
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
wine_hrefs = []
wine_brand_names = []
wine_product_names = []
driver.get(vivino_page)

# Select "Australia" as ship to location
select_ship_to_country(driver, "Australia")
                      
#expand price slider - skipping this as slider seems to update automatically whenever country is changed to australia
#expand_slider(driver)

wine_cards = driver.find_elements(By.CLASS_NAME, "card__card--2R5Wh")
foot_header = driver.find_element(By.CLASS_NAME, "footer_header__cq48I")
n_wine_cards = len(wine_cards)
previous_n_wine_cards = n_wine_cards
#previous_last_result = wine_cards[-1].text
while n_wine_cards < n_to_scroll:
    print(f"{n_wine_cards} wine results found. Continuing to scroll...")
    scroll_to_element(foot_header)

    # check results have updated
    wait_counter = 0
    wait_limit = 5
    while True:
        wine_cards = driver.find_elements(By.CLASS_NAME, "card__card--2R5Wh")
        #current_last_result = wine_cards[-1].text_length
        n_wine_cards = len(wine_cards)
        if wait_counter >= wait_limit:
            print("More results not loading. Trying to scrolling up then down again...")
            wait_counter = 0
            scroll_to_element(wine_cards[-10])
            time.sleep(5)
            scroll_to_element(foot_header)
        #if current_last_result == previous_last_result:
        print(f"Comparing current results count:{n_wine_cards} with previous results cout:{previous_n_wine_cards}")
        if n_wine_cards == previous_n_wine_cards:
            print("Results not updated. Waiting and retrieving again...")
            wait_counter += 1
            time.sleep(1)
            continue
        else:
            #n_wine_cards = len(wine_cards)
            #previous_last_result = current_last_result
            previous_n_wine_cards = n_wine_cards
            break
       
print(f"{n_wine_cards} wine results found. Stopping scrolling and retrieving results...")


Successfully selected Australia from the dropdown.
25 wine results found. Continuing to scroll...
Comparing current results count:25 with previous results cout:25
Results not updated. Waiting and retrieving again...
Comparing current results count:25 with previous results cout:25
Results not updated. Waiting and retrieving again...
Comparing current results count:50 with previous results cout:25
50 wine results found. Continuing to scroll...
Comparing current results count:50 with previous results cout:50
Results not updated. Waiting and retrieving again...
Comparing current results count:50 with previous results cout:50
Results not updated. Waiting and retrieving again...
Comparing current results count:75 with previous results cout:50
75 wine results found. Continuing to scroll...
Comparing current results count:75 with previous results cout:75
Results not updated. Waiting and retrieving again...
Comparing current results count:75 with previous results cout:75
Results not updated. Wa

In [ ]:
product_hrefs = []
brand_names = []
product_names = []
#for i in range(len(wine_cards)):
for i in range(i_start, i_end, 1):
    print(f"Retriving data from result i={i} of {n_wine_cards-1}")
    product_anchor = wine_cards[i].find_element(By.CLASS_NAME, "anchor_anchor__m8Qi-")
    product_hrefs.append(product_anchor.get_attribute("href"))
    brand_and_product_names = wine_cards[i].find_elements(By.CLASS_NAME, "wineInfoVintage__truncate--3QAtw")
    brand_names.append(brand_and_product_names[0].text)
    product_names.append(brand_and_product_names[1].text)
    time.sleep(0.1)

vivino_product_hrefs_df = pd.DataFrame({
    "product_href" : product_hrefs,
    "brand_name" : brand_names ,
    "product_name" : product_names,
})

Retriving data from result i=0 of 999
Retriving data from result i=1 of 999
Retriving data from result i=2 of 999
Retriving data from result i=3 of 999
Retriving data from result i=4 of 999
Retriving data from result i=5 of 999
Retriving data from result i=6 of 999
Retriving data from result i=7 of 999
Retriving data from result i=8 of 999
Retriving data from result i=9 of 999
Retriving data from result i=10 of 999
Retriving data from result i=11 of 999
Retriving data from result i=12 of 999
Retriving data from result i=13 of 999
Retriving data from result i=14 of 999
Retriving data from result i=15 of 999
Retriving data from result i=16 of 999
Retriving data from result i=17 of 999
Retriving data from result i=18 of 999
Retriving data from result i=19 of 999
Retriving data from result i=20 of 999
Retriving data from result i=21 of 999
Retriving data from result i=22 of 999
Retriving data from result i=23 of 999
Retriving data from result i=24 of 999
Retriving data from result i=25 of 

In [ ]:
vivino_product_hrefs_df

In [ ]:
file_path = f"../data/vivino/popular_vivino_subsets/popular_vivino_hrefs_{i_start}_{i_end-1}.csv"
vivino_product_hrefs_df.to_csv(file_path, index=False)

In [ ]:
driver.quit()